# Olympics — The Hidden Costs: Host-City Debt Map

An editable Altair map of every Olympic host city (1976–2024), sized by total cost and
coloured by how the debt was ultimately resolved.

**Data:** `data/olympic_host_debt_starter.csv`

This notebook is the working draft for the map that goes into the scrollytelling report
(`site/report.html`). Tweak the encodings below, then run the **Export** cell at the bottom
to drop it straight into `site/charts/`.

## 1. Load the data & attach coordinates

The CSV has no lat/lon, so we attach a small lookup. Edit `COORDS` if you add or move a city.

In [1]:
# --- Setup: runs on Colab (self-contained) AND locally ---
try:
    import vega_datasets  # not preinstalled on Colab
except ImportError:
    %pip install -q vega_datasets

import os, io, base64
import altair as alt
import pandas as pd
from vega_datasets import data

# Use the local CSV if present (your machine); otherwise use the
# copy embedded below, so the notebook is fully self-contained in Colab.
LOCAL = "data/olympic_host_debt_starter.csv"
_CSV_B64 = (
    "Z2FtZXMseWVhcix0eXBlLGhvc3RfY2l0eSxob3N0X2NvdW50cnkscmVnaW9uX2NhcnJ5aW5nX2RlYnQsdG90YWxfY29zdF91c2Rf"
    "Ym5fbm9taW5hbCxjb3N0X292ZXJydW5fcGN0LHB1YmxpY19zaGFyZV9wY3QsZGVidF9zdGF0dXMsZGVidF9wYWlkX29mZl95ZWFy"
    "LHllYXJzX3RvX3JlcGF5LHN0aWxsX3BheWluZ19ub3RlcyxwcmltYXJ5X3NvdXJjZV91cmwsc2Vjb25kYXJ5X3NvdXJjZV91cmwK"
    "MTk3NiBTdW1tZXIsMTk3NixzdW1tZXIsTW9udHJlYWwsQ2FuYWRhLFF1ZWJlYyAocHJvdmluY2UpLDEuNSw3MjAsMTAwLHBhaWRf"
    "b2ZmLDIwMDYsMzAsIlF1ZWJlYyBzcGVjaWFsIHRvYmFjY28gdGF4IGZ1bmRlZCByZXBheW1lbnQ7IGZpbmFsIHBheW1lbnQgTm92"
    "ZW1iZXIgMjAwNiAofkMkMS42MUIgdG90YWwgaW5jLiBpbnRlcmVzdCAmIHJlcGFpcnMpLiBDb2luZWQgIiJUaGUgQmlnIE93ZSIi"
    "LiIsaHR0cHM6Ly93ZWIuYXJjaGl2ZS5vcmcvd2ViLzIwMDcwMTAzMTAzNDI2L2h0dHBzOi8vd3d3LmNiYy5jYS9uZXdzL2NhbmFk"
    "YS9tb250cmVhbC9xdWViZWMtcy1iaWctb3dlLXN0YWRpdW0tZGVidC1pcy1vdmVyLTEuNjAyNTMwLGh0dHBzOi8vd3d3LnRoZWd1"
    "YXJkaWFuLmNvbS9jaXRpZXMvMjAxNi9qdWwvMDYvNDAteWVhci1oYW5nb3Zlci0xOTc2LW9seW1waWMtZ2FtZXMtYnJva2UtbW9u"
    "dHJlYWwtY2FuYWRhCjE5ODAgV2ludGVyLDE5ODAsd2ludGVyLExha2UgUGxhY2lkLFVTQSxOZXcgWW9yayBTdGF0ZSwsMzI0LCxm"
    "ZWRlcmFsX2JhaWxvdXQsLCwiQ29uZ3Jlc3Npb25hbCBiYWlsb3V0IG9mIExha2UgUGxhY2lkIE9yZ2FuaXppbmcgQ29tbWl0dGVl"
    "OyBVU09DICYgZmVkZXJhbCBmdW5kcyBjb3ZlcmVkIHNob3J0ZmFsbC4iLGh0dHBzOi8vd3d3Lmdhby5nb3YvcHJvZHVjdHMvZ2dk"
    "LTAwLTE4MywKMTk4MCBTdW1tZXIsMTk4MCxzdW1tZXIsTW9zY293LFVTU1IsVVNTUiAoU292aWV0IFVuaW9uKSw5LjAsNDIsMTAw"
    "LGRpc3NvbHZlZF9zdGF0ZSwsLCJEZWJ0IGFic29yYmVkIGJ5IFVTU1IgZmVkZXJhbCBidWRnZXQ7IGhhcmQgdG8gaXNvbGF0ZSBw"
    "b3N0LTE5OTEgc3VjY2Vzc29yIG9ibGlnYXRpb25zLiIsaHR0cHM6Ly9lbi53aWtpcGVkaWEub3JnL3dpa2kvMTk4MF9TdW1tZXJf"
    "T2x5bXBpY3MsCjE5ODQgV2ludGVyLDE5ODQsd2ludGVyLFNhcmFqZXZvLFl1Z29zbGF2aWEsU1IgQm9zbmlhIGFuZCBIZXJ6ZWdv"
    "dmluYSwsLCwsLCwgIkJvc25pYW4gV2FyICgxOTkyLTk1KSBkZXN0cm95ZWQgbW9zdCB2ZW51ZXM7IG5vIGZvcm1hbCBkZWJ0LXJl"
    "cGF5bWVudCByZWNvcmQuIixodHRwczovL2VuLndpa2lwZWRpYS5vcmcvd2lraS8xOTg0X1dpbnRlcl9PbHltcGljcywKMTk4NCBT"
    "dW1tZXIsMTk4NCxzdW1tZXIsTG9zIEFuZ2VsZXMsVVNBLENpdHkgb2YgTG9zIEFuZ2VsZXMsMC43MiwsMCxzdXJwbHVzLCwwLCJG"
    "aXJzdCBmdWxseSBwcml2YXRlbHktZmluYW5jZWQgbW9kZXJuIEdhbWVzOyBVU0QgMjE1LTI1ME0gc3VycGx1cywgbm8gcHVibGlj"
    "IGRlYnQuIixodHRwczovL3d3dy5sYXRpbWVzLmNvbS9hcmNoaXZlcy9sYS14cG0tMTk4NS0wNy0zMS1zcC01NDA1LXN0b3J5Lmh0"
    "bWwsCjE5ODggV2ludGVyLDE5ODgsd2ludGVyLENhbGdhcnksQ2FuYWRhLEFsYmVydGEgKHByb3ZpbmNlKSwsNTksLHBhaWRfb2Zm"
    "X3dpdGhfc3VycGx1cywxOTg4LDAsIk9DTyc4OCBjbG9zZWQgd2l0aCBDJDkwTSBzdXJwbHVzOyBlbmRvd2VkIENhbGdhcnkgT2x5"
    "bXBpYyBEZXZlbG9wbWVudCBBc3NvY2lhdGlvbiAoc3RpbGwgb3BlcmF0aW5nIHRvZGF5KS4iLGh0dHBzOi8vZW4ud2lraXBlZGlh"
    "Lm9yZy93aWtpLzE5ODhfV2ludGVyX09seW1waWNzLAoxOTg4IFN1bW1lciwxOTg4LHN1bW1lcixTZW91bCxTb3V0aCBLb3JlYSxT"
    "b3V0aCBLb3JlYSw0LjAsLTE0LCxzdXJwbHVzLCwwLCJSZXBvcnRlZCB+VVNEIDMwME0rIHN1cnBsdXM7IHVzZWQgdG8gc2VlZCBL"
    "b3JlYSBTcG9ydHMgUHJvbW90aW9uIEZvdW5kYXRpb24uIixodHRwczovL3d3dy5sYXRpbWVzLmNvbS9hcmNoaXZlcy9sYS14cG0t"
    "MTk4OS0wMy0yOS1zcC03NjEtc3RvcnkuaHRtbCwKMTk5MiBXaW50ZXIsMTk5Mix3aW50ZXIsQWxiZXJ0dmlsbGUsRnJhbmNlLFNh"
    "dm9pZSAoZMOpcGFydGVtZW50KSwsMTM1LCxsb25nX2RlYnQsLDE3LCJTYXZvaWUgZMOpcGFydGVtZW50IHJlcXVpcmVkIHJlZ2lv"
    "bmFsICYgbmF0aW9uYWwgYmFpbG91dHMgdGhyb3VnaCB0aGUgMTk5MHM7IGRlYnQgcmVwb3J0ZWQgc3RpbGwgb24gcmVnaW9uYWwg"
    "Ym9va3MgdGhyb3VnaCAyMDAwcy4iLGh0dHBzOi8vZW4ud2lraXBlZGlhLm9yZy93aWtpLzE5OTJfV2ludGVyX09seW1waWNzLAox"
    "OTkyIFN1bW1lciwxOTkyLHN1bW1lcixCYXJjZWxvbmEsU3BhaW4sQ2F0YWxvbmlhICsgU3BhaW4gKGZlZGVyYWwpLDkuMywyNjYs"
    "NjEsbG9uZ19kZWJ0LCwyMCwiUHVibGljIHNoYXJlIH42MSUgKGNpdHkgMjAlLCByZWdpb24gMjAlLCBuYXRpb25hbCAyMSUpLiBD"
    "aXR5ICYgbmF0aW9uYWwgZGVidCBhdHRyaWJ1dGVkIHRvIEdhbWVzIHNlcnZpY2VkIGludG8gdGhlIDIwMTBzLiIsaHR0cHM6Ly9k"
    "ZGQudWFiLmNhdC9wdWIvd29ycGFwLzE5OTUvOTUwMjgvZWNvYW5hb2Z0aGViYV9hMTk5NWlzby5wZGYsCjE5OTQgV2ludGVyLDE5"
    "OTQsd2ludGVyLExpbGxlaGFtbWVyLE5vcndheSxOb3J3YXkgKG5hdGlvbmFsKSwxLjcsMjc3LCxwYWlkX29mZiwsLCJDZW50cmFs"
    "LWdvdmVybm1lbnQgZnVuZGVkOyBubyBkZWRpY2F0ZWQgT2x5bXBpYyBkZWJ0IHZlaGljbGUgc3Vydml2ZXMuIixodHRwczovL2Vu"
    "Lndpa2lwZWRpYS5vcmcvd2lraS8xOTk0X1dpbnRlcl9PbHltcGljcywKMTk5NiBTdW1tZXIsMTk5NixzdW1tZXIsQXRsYW50YSxV"
    "U0EsQUNPRyAocHJpdmF0ZSkgKyBDaXR5IG9mIEF0bGFudGEsMS44LDE0NywwLHBhaWRfb2ZmX3dpdGhfc3VycGx1cywsMCwiQUNP"
    "RyBvcGVyYXRpb25hbGx5IGJhbGFuY2VkOyByZXNpZHVhbCBwdWJsaWMgaW5mcmFzdHJ1Y3R1cmUgZGVidCBzbWFsbCB2cy4gdG90"
    "YWwuIixodHRwczovL2VuLndpa2lwZWRpYS5vcmcvd2lraS8xOTk2X1N1bW1lcl9PbHltcGljcywKMTk5OCBXaW50ZXIsMTk5OCx3"
    "aW50ZXIsTmFnYW5vLEphcGFuLE5hZ2FubyBQcmVmZWN0dXJlLDEyLjUsNTYsLHBhaWRfb2ZmLDIwMTUsMTcsIk5hZ2FubyBQcmVm"
    "ZWN0dXJlIGJvbmRzIGZvciBTaGlua2Fuc2VuICYgdmVudWVzIHNlcnZpY2VkIGludG8gbWlkLTIwMTBzOyB+SlBZIDEwMEIgcmVt"
    "YWluaW5nIGFzIG9mIDIwMDguIixodHRwczovL2VuLndpa2lwZWRpYS5vcmcvd2lraS8xOTk4X1dpbnRlcl9PbHltcGljcyxodHRw"
    "czovL3d3dy5qYXBhbnRpbWVzLmNvLmpwL25ld3MvMjAxMy8wMi8yNC9uYXRpb25hbC9uYWdhbm8tc3RpbGwtcGF5aW5nLW9seW1w"
    "aWNzLWRlYnQvCjIwMDAgU3VtbWVyLDIwMDAsc3VtbWVyLFN5ZG5leSxBdXN0cmFsaWEsTmV3IFNvdXRoIFdhbGVzIChzdGF0ZSks"
    "NC41LDkwLCxsb25nX2RlYnQsLDEwLCJOU1cgQXVkaXRvci1HZW5lcmFsIHJlcG9ydGVkIH5BVUQgMi40QiBuZXQgcHVibGljIGNv"
    "c3Q7IHZlbnVlIG9wZXJhdGluZyBsb3NzZXMgY29udGludWVkIDEwKyB5ZWFycy4iLGh0dHBzOi8vd3d3LnRlbGVncmFwaC5jby51"
    "ay9zcG9ydC9vbHltcGljcy8yMzA3NDI2L0xvbmRvbi0yMDEyLW11c3QtbGVhcm4tZnJvbS10aGUtMWJuLVN5ZG5leS1oYW5nb3Zl"
    "ci5odG1sLAoyMDAyIFdpbnRlciwyMDAyLHdpbnRlcixTYWx0IExha2UgQ2l0eSxVU0EsVXRhaCArIFVTIGZlZGVyYWwsMS45LDI5"
    "LCxwYWlkX29mZl93aXRoX3N1cnBsdXMsLDAsIlVTRCB+VVNEIDEwME0gc3VycGx1czsgZmVkZXJhbCBjb250cmlidXRpb24gflVT"
    "RCAxLjNCIChHQU8gMjAwMSkuIixodHRwczovL3d3dy5nYW8uZ292L3Byb2R1Y3RzL2dhby0wMS0xMTQxdCwKMjAwNCBTdW1tZXIs"
    "MjAwNCxzdW1tZXIsQXRoZW5zLEdyZWVjZSxHcmVlY2UgKG5hdGlvbmFsKSwxNS4wLDQ5LCxzdGlsbF9wYXlpbmcsLDIwKywiR3Jl"
    "ZWsgc3RhdGUgY2FycmllcyB0aGUgT2x5bXBpYyBjb3N0IGFzIHBhcnQgb2YgaXRzIHB1YmxpYy1kZWJ0IG92ZXJoYW5nIGNpdGVk"
    "IGluIFRyb2lrYS9JTUYgcHJvZ3JhbW1lczsgfkVVUiA5QiBhZGRlZCB0byBuYXRpb25hbCBkZWJ0IChCYW5rIG9mIEdyZWVjZSBl"
    "c3RpbWF0ZSkuIENvbnNpZGVyZWQgY29udHJpYnV0aW5nIGZhY3RvciB0byAyMDEwLTIwMTggZGVidCBjcmlzaXMuIixodHRwczov"
    "L3d3dy5yZXV0ZXJzLmNvbS9hcnRpY2xlL3VzLWdyZWVjZS1vbHltcGljcy1pZFVTS0JOMEc3MERQMjAxNDA4MDcsaHR0cHM6Ly93"
    "d3cuYmFua29mZ3JlZWNlLmdyL2VuL3N0YXRpc3RpY3MvZXh0ZXJuYWwtc2VjdG9yL2JhbGFuY2Utb2YtcGF5bWVudHMKMjAwNiBX"
    "aW50ZXIsMjAwNix3aW50ZXIsVHVyaW4sSXRhbHksUGllbW9udGUgKHJlZ2lvbikgKyBJdGFseSAobmF0aW9uYWwpLDQuNCw4Miws"
    "bG9uZ19kZWJ0LCwxNSwiVHVyaW4gT3JnYW5pemluZyBDb21taXR0ZWUgY2xvc2VkIHdpdGggfkVVUiAzNE0gZGVmaWNpdDsgUGll"
    "bW9udGUgc2VydmljaW5nIHZlbnVlLW1haW50ZW5hbmNlIGRlYnQgdGhyb3VnaCAyMDIwcyAocGVyIENvcnRlIGRlaSBDb250aSku"
    "IixodHRwczovL3d3dy5jb3J0ZWNvbnRpLml0LywKMjAwOCBTdW1tZXIsMjAwOCxzdW1tZXIsQmVpamluZyxDaGluYSxDaGluYSAo"
    "bmF0aW9uYWwpLDQ0LjAsNCwsb3BhcXVlLCwsIkNoaW5lc2UgTmF0aW9uYWwgQXVkaXQgT2ZmaWNlIDIwMDkgcmVwb3J0OyBjZW50"
    "cmFsICYgQmVpamluZyBtdW5pY2lwYWwgZnVuZGluZyBub3QgYnJva2VuIG91dCBwdWJsaWNseS4iLGh0dHA6Ly9lbmdsaXNoLnBy"
    "YXZkYS5ydS9zcG9ydHMvZ2FtZXMvMDYtMDgtMjAwOC8xMDYwMDMtYmVpamluZ19vbHltcGljcy0wLywKMjAxMCBXaW50ZXIsMjAx"
    "MCx3aW50ZXIsVmFuY291dmVyLENhbmFkYSxCcml0aXNoIENvbHVtYmlhIChwcm92aW5jZSksMi41LDE3LCxwYWlkX29mZiwyMDE0"
    "LDQsIlZBTk9DIGRlY2xhcmVkIGRlYnQtZnJlZSAyMDE0OyBCQyBwcm92aW5jaWFsIHNoYXJlIH5DQUQgOTI1TSBhYnNvcmJlZC4i"
    "LGh0dHBzOi8vd3d3LmNiYy5jYS9uZXdzL2NhbmFkYS9icml0aXNoLWNvbHVtYmlhL3ZhbmNvdXZlci0yMDEwLXdpbnRlci1vbHlt"
    "cGljcy1kZWJ0LWZyZWUtdmFub2MtZmluYWwtcmVwb3J0LXNheXMtMS4yNjk1OTk0LAoyMDEyIFN1bW1lciwyMDEyLHN1bW1lcixM"
    "b25kb24sVW5pdGVkIEtpbmdkb20sVUsgKG5hdGlvbmFsKSArIEdMQSwxNS4wLDc2LDgwLHBhaWRfb2ZmLDIwMTMsMSwiUHVibGlj"
    "IFNlY3RvciBGdW5kaW5nIFBhY2thZ2UgR0JQIDkuM0I7IE5hdGlvbmFsIEF1ZGl0IE9mZmljZSBmaW5hbCByZXBvcnQgMjAxMi4i"
    "LGh0dHBzOi8vd3d3Lm5hby5vcmcudWsvd3AtY29udGVudC91cGxvYWRzLzIwMTIvMTIvMTIxMzgxMi5wZGYsCjIwMTQgV2ludGVy"
    "LDIwMTQsd2ludGVyLFNvY2hpLFJ1c3NpYSxSdXNzaWEgKGZlZGVyYWwpICsgVkVCLlJGLDU1LjAsMjg5LCxzdGlsbF9wYXlpbmcs"
    "LDEwKywiU3RhdGUgYmFuayBWRUIuUkYgaG9sZHMgflJVQiAyNDBCIG9mIG5vbi1wZXJmb3JtaW5nIFNvY2hpIE9seW1waWMgbG9h"
    "bnMgYXMgb2YgMjAyNCAoUmV1dGVycywgVmVkb21vc3RpKS4gRmVkZXJhbCBidWRnZXQgY29udGludWVzIHRvIHNlcnZpY2UuIixo"
    "dHRwczovL3d3dy5yZXV0ZXJzLmNvbS93b3JsZC9ldXJvcGUvcnVzc2lhcy12ZWJyZi1zdGlsbC13cml0aW5nLW9mZi1zb2NoaS1v"
    "bHltcGljLWxvYW5zLTIwMjItMDQtMTUvLGh0dHBzOi8vd3d3LnRoZW1vc2Nvd3RpbWVzLmNvbS8yMDIwLzAyLzA3L3NpeC15ZWFy"
    "cy1vbi1zb2NoaS1zdGlsbC1zdHJ1Z2dsZXMtd2l0aC1vbHltcGljLWRlYnRzLWE2OTE4NQoyMDE2IFN1bW1lciwyMDE2LHN1bW1l"
    "cixSaW8gZGUgSmFuZWlybyxCcmF6aWwsUmlvIGRlIEphbmVpcm8gU3RhdGUsMTMuMSwzNTIsLHN0aWxsX3BheWluZywsMTArLCJT"
    "dGF0ZSBvZiBSaW8gZGVjbGFyZWQgZmluYW5jaWFsIGVtZXJnZW5jeSAoRGVjcmVlIDQ1LjY5Mi8yMDE2KSB0d28gbW9udGhzIGJl"
    "Zm9yZSBHYW1lczsgYXVzdGVyaXR5ICYgZmVkZXJhbCBiYWlsb3V0IHRocm91Z2ggMjAyMHMuIE11c2V1bS1vZi1Ub21vcnJvdyAm"
    "IHZlbnVlIG1haW50ZW5hbmNlIGRlYnRzIHVucmVzb2x2ZWQuIixodHRwczovL29seW1waWNzLm5iY3Nwb3J0cy5jb20vMjAxNy8w"
    "Ni8xNC9yaW8tb2x5bXBpY3MtY29zdC0xMy1iaWxsaW9uLyxodHRwczovL3d3dy5yZXV0ZXJzLmNvbS9hcnRpY2xlL3VzLW9seW1w"
    "aWNzLTIwMTYtcmlvLWVtZXJnZW5jeS1pZFVTS0NOMFlXMkJGCjIwMTggV2ludGVyLDIwMTgsd2ludGVyLFB5ZW9uZ0NoYW5nLFNv"
    "dXRoIEtvcmVhLEdhbmd3b24gUHJvdmluY2UsMTIuOSwsLGxvbmdfZGVidCwsNSssIkdhbmd3b24gUHJvdmluY2lhbCBHb3Zlcm5t"
    "ZW50IHJlcG9ydGVkIH5LUlcgOTZCL3llYXIgdmVudWUgb3BlcmF0aW5nIGxvc3NlcyB0aHJvdWdoIDIwMjM7IGNlbnRyYWwtZ292"
    "ZXJubWVudCBzdWJzaWRpZXMgb25nb2luZy4iLGh0dHBzOi8vZW4ueW5hLmNvLmtyL3ZpZXcvQUVOMjAyMzAxMjMwMDY0MDAzMTUs"
    "CjIwMjAgU3VtbWVyLDIwMjEsc3VtbWVyLFRva3lvLEphcGFuLFRva3lvIE1ldHJvcG9saXRhbiBHb3YgKyBKYXBhbiAobmF0aW9u"
    "YWwpLDEzLjAsMjgzLDYwLGxvbmdfZGVidCwsMTArLCJCb2FyZCBvZiBBdWRpdCBvZiBKYXBhbiAyMDIyIGZpbmFsIHJlcG9ydDog"
    "dG90YWwgSlBZIDEuNDJUICh+VVNEIDEzQik7IFRva3lvICYgbmF0aW9uYWwgZ292ZXJubWVudCBzcGxpdCA2MC80MCBvZiBwdWJs"
    "aWMgc2hhcmUuIERlYnQtc2VydmljZSB0YWlsIHByb2plY3RlZCB0aHJvdWdoIDIwMzBzLiIsaHR0cHM6Ly93d3cuamJhdWRpdC5n"
    "by5qcC9lbmdsaXNoLyxodHRwczovL3d3dy5yZXV0ZXJzLmNvbS9saWZlc3R5bGUvc3BvcnRzL3Rva3lvLW9seW1waWNzLW9mZmlj"
    "aWFsLWNvc3Qtd2FzLTEzLWJpbGxpb24tcmVwb3J0LTIwMjItMTItMjEvCjIwMjIgV2ludGVyLDIwMjIsd2ludGVyLEJlaWppbmcs"
    "Q2hpbmEsQ2hpbmEgKG5hdGlvbmFsKSwzLjksLCxvcGFxdWUsLCwiU3RhdGUtZmluYW5jZWQ7IG5vIHB1YmxpYyBicmVha2Rvd24g"
    "b2YgT2x5bXBpYyBkZWJ0LiIsaHR0cHM6Ly9lbi53aWtpcGVkaWEub3JnL3dpa2kvMjAyMl9XaW50ZXJfT2x5bXBpY3MsCjIwMjQg"
    "U3VtbWVyLDIwMjQsc3VtbWVyLFBhcmlzLEZyYW5jZSxGcmFuY2UgKG5hdGlvbmFsKSArIMOObGUtZGUtRnJhbmNlLDkuNywxMTUs"
    "NjAscmVjZW50LCwsIkZyZW5jaCBDb3VyIGRlcyBjb21wdGVzIEp1bmUgMjAyNTogRVVSIDZCIHRheHBheWVyIGNvbnRyaWJ1dGlv"
    "bjsgT0NPRyBjbG9zZWQgd2l0aCBFVVIgNzZNIHN1cnBsdXMuIixodHRwczovL3d3dy5sZW1vbmRlLmZyL2VuL3Nwb3J0cy9hcnRp"
    "Y2xlLzIwMjUvMDYvMjMvcGFyaXMtb2x5bXBpY3MtYW5kLXBhcmFseW1waWNzLWNvc3QtdGF4cGF5ZXJzLW5lYXJseS02LWJpbGxp"
    "b25fNjc0MjYyOF85Lmh0bWwsaHR0cHM6Ly93d3cuY2NvbXB0ZXMuZnIvCg=="
)
df = (pd.read_csv(LOCAL) if os.path.exists(LOCAL)
      else pd.read_csv(io.StringIO(base64.b64decode(_CSV_B64).decode())))

COORDS = {
    "Montreal": (45.50, -73.57),      "Lake Placid": (44.28, -73.98),
    "Moscow": (55.76, 37.62),         "Sarajevo": (43.85, 18.36),
    "Los Angeles": (34.05, -118.24),  "Calgary": (51.05, -114.07),
    "Seoul": (37.57, 126.98),         "Albertville": (45.68, 6.39),
    "Barcelona": (41.39, 2.17),       "Lillehammer": (61.12, 10.47),
    "Atlanta": (33.75, -84.39),       "Nagano": (36.65, 138.18),
    "Sydney": (-33.87, 151.21),       "Salt Lake City": (40.76, -111.89),
    "Athens": (37.98, 23.73),         "Turin": (45.07, 7.69),
    "Beijing": (39.90, 116.41),       "Vancouver": (49.28, -123.12),
    "London": (51.51, -0.13),         "Sochi": (43.60, 39.73),
    "Rio de Janeiro": (-22.91, -43.17), "PyeongChang": (37.37, 128.39),
    "Tokyo": (35.68, 139.69),         "Paris": (48.86, 2.35),
}

df["lat"] = df["host_city"].map(lambda c: COORDS[c][0])
df["lon"] = df["host_city"].map(lambda c: COORDS[c][1])

# Human-readable label for the debt outcome (used for colour + legend)
DEBT_LABELS = {
    "surplus": "Surplus / no public debt",
    "paid_off_with_surplus": "Surplus / no public debt",
    "paid_off": "Paid off",
    "long_debt": "Long / lingering debt",
    "still_paying": "Still paying",
    "federal_bailout": "Bailout",
    "dissolved_state": "Bailout",
    "opaque": "Opaque / unreported",
    "recent": "Too recent",
}
df["debt_outcome"] = df["debt_status"].map(DEBT_LABELS).fillna("Unknown")

df[["games", "host_city", "total_cost_usd_bn_nominal", "cost_overrun_pct", "debt_status", "debt_outcome"]]

,games,host_city,total_cost_usd_bn_nominal,cost_overrun_pct,debt_status,debt_outcome
0,1976 Summer,Montreal,1.50,720.0,paid_off,Paid off
1,1980 Winter,Lake Placid,NaN,324.0,federal_bailout,Bailout
2,1980 Summer,Moscow,9.00,42.0,dissolved_state,Bailout
3,1984 Winter,Sarajevo,NaN,NaN,NaN,Unknown
4,1984 Summer,Los Angeles,0.72,NaN,surplus,Surplus / no public debt
5,1988 Winter,Calgary,NaN,59.0,paid_off_with_surplus,Surplus / no public debt
6,1988 Summer,Seoul,4.00,-14.0,surplus,Surplus / no public debt
7,1992 Winter,Albertville,NaN,135.0,long_debt,Long / lingering debt
8,1992 Summer,Barcelona,9.30,266.0,long_debt,Long / lingering debt
9,1994 Winter,Lillehammer,1.70,277.0,paid_off,Paid off


## 2. Compare the top 5 host countries

Before looking at *where* the Games were held, compare the biggest spenders on three axes:
**total budget**, **average cost overrun**, and **how long the host carried Olympic debt**.
Countries are ranked by total nominal budget (summed across each country's Games).

> Budgets are **nominal USD** and mix sports-only vs. total-infrastructure figures, so read
> them as orders of magnitude. China has no public debt-repayment record (opaque → no bar),
> and South Korea's overrun is negative (Seoul 1988 came in under budget).

In [2]:
import numpy as np

# --- Aggregate to country level ---------------------------------------------
# budget  = total nominal cost summed over that country's Games
# overrun = mean cost-overrun %
# debt    = longest years-to-repay (parsing "20+" -> 20)
df["years_to_repay_num"] = df["years_to_repay"].apply(
    lambda v: np.nan if pd.isna(v) else float(str(v).replace("+", "").strip() or "nan")
)
by_country = (
    df.groupby("host_country")
    .agg(
        budget=("total_cost_usd_bn_nominal", "sum"),
        overrun=("cost_overrun_pct", "mean"),
        debt_years=("years_to_repay_num", "max"),
    )
    .reset_index()
    .sort_values("budget", ascending=False)
)

top5 = by_country.head(5).copy()
order = top5["host_country"].tolist()

METRICS = {
    "budget": "Total budget (US$ bn)",
    "overrun": "Avg cost overrun (%)",
    "debt_years": "Years carrying debt",
}
long = top5.melt(
    id_vars="host_country", value_vars=list(METRICS),
    var_name="metric", value_name="value",
)
long["metric"] = long["metric"].map(METRICS)

compare = (
    alt.Chart(long)
    .mark_bar()
    .encode(
        y=alt.Y("host_country:N", sort=order, title=None),
        x=alt.X("value:Q", title=None),
        color=alt.Color("host_country:N", sort=order, legend=None,
                        scale=alt.Scale(scheme="tableau10")),
        tooltip=["host_country:N", "metric:N", alt.Tooltip("value:Q", format=".1f")],
    )
    .properties(width=200, height=170)
    .facet(column=alt.Column("metric:N", sort=list(METRICS.values()), title=None))
    .resolve_scale(x="independent")
    .properties(title="Top 5 host countries by total Olympic budget — cost, overrun & debt")
)
compare

/Users/flo/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Users/flo/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.FacetChart(...)

## 3. Build the map

- **Position** — host city on a Natural Earth world projection
- **Size** — total cost of the Games (US$ bn, nominal)
- **Colour** — how the debt was ultimately resolved

Change the projection (`"naturalEarth1"`, `"equalEarth"`, `"mercator"`…), the `size`/`color`
fields, or the palette to make it your own.

In [3]:
countries = alt.topo_feature(data.world_110m.url, "countries")

# Legend order (best -> worst outcome) and matching semantic colours
OUTCOME_ORDER = [
    "Surplus / no public debt", "Paid off", "Long / lingering debt",
    "Still paying", "Bailout", "Opaque / unreported", "Too recent", "Unknown",
]
OUTCOME_COLORS = [
    "#2ca02c",  # Surplus / no public debt  -> green
    "#2ca02c",  # Paid off                  -> green
    "#ff7f0e",  # Long / lingering debt     -> orange
    "#d62728",  # Still paying              -> red
    "#000000",  # Bailout                   -> black
    "#9e9e9e",  # Opaque / unreported       -> grey
    "#9e9e9e",  # Too recent                -> grey
    "#9e9e9e",  # Unknown                   -> grey
]

ocean = alt.Chart(alt.sphere()).mark_geoshape(fill="#eaf3f9")
land = alt.Chart(countries).mark_geoshape(fill="#e0e0e0", stroke="white", strokeWidth=0.5)

cities = (
    alt.Chart(df)
    .mark_circle(opacity=0.8, stroke="white", strokeWidth=0.6)
    .encode(
        longitude="lon:Q",
        latitude="lat:Q",
        size=alt.Size(
            "total_cost_usd_bn_nominal:Q",
            title="Total cost (US$ bn)",
            scale=alt.Scale(range=[30, 1200]),
        ),
        color=alt.Color(
            "debt_outcome:N",
            title="Debt outcome",
            sort=OUTCOME_ORDER,
            scale=alt.Scale(domain=OUTCOME_ORDER, range=OUTCOME_COLORS),
        ),
        tooltip=[
            alt.Tooltip("games:N", title="Games"),
            alt.Tooltip("host_city:N", title="Host city"),
            alt.Tooltip("host_country:N", title="Country"),
            alt.Tooltip("total_cost_usd_bn_nominal:Q", title="Total cost (US$ bn)"),
            alt.Tooltip("cost_overrun_pct:Q", title="Cost overrun (%)"),
            alt.Tooltip("debt_outcome:N", title="Debt outcome"),
        ],
    )
)

olympic_map = (
    (ocean + land + cities)
    .project("naturalEarth1")
    .properties(width=820, height=440, title="Olympic host cities — the hidden costs (1976–2024)")
    .configure_view(stroke=None)
)
olympic_map

/Users/flo/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Users/flo/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Users/flo/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Users/flo/anaconda3/lib/python3.12/site

alt.LayerChart(...)

## 4. Export to the report site

Uncomment the format you want. Use `width="container"` so the chart resizes to fit the report
column, then wire it into `site/main.js` (e.g. as `chart_2.html` for step 2).

In [4]:
# Rebuild at container width for the responsive site layout
export_map = (
    (ocean + land + cities)
    .project("naturalEarth1")
    .properties(width="container", height=440,
                title="Olympic host cities — the hidden costs (1976–2024)")
    .configure_view(stroke=None)
)

# Option A — standalone HTML (good for maps; loaded via <iframe> in main.js)
# export_map.save("site/charts/chart_2.html")

# Option B — Vega-Lite JSON (inline via vega-embed in main.js)
# export_map.save("site/charts/chart_0.json")

print("Uncomment a .save(...) line above to export into site/charts/")

Uncomment a .save(...) line above to export into site/charts/
